In [1]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.schema import Document
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
llm = ChatOpenAI(model = "gpt-3.5-turbo")
embeddings = OpenAIEmbeddings()

## State Definition

In [3]:
class AgentState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    needs_retrieval: bool

In [ ]:
sample_texts = [
    "LangGraph is a Python framework for building stateful, multi-step LLM workflows where each node represents an agent, tool, or function connected through a directed graph. It lets you design complex reasoning pipelines—like supervisors, workers, and tool calls—with deterministic control over execution flow.",
    "RAG (Retrieval-Augmented Generation) combines an LLM with an external knowledge store so the model retrieves relevant documents before generating an answer. This dramatically improves accuracy by grounding responses in real data instead of relying solely on the model's internal training.",
    "Vector databases store text, images, or other data as high-dimensional embeddings and allow fast similarity search using algorithms like HNSW or IVF. They power semantic search, RAG pipelines, clustering, and recommendation systems by retrieving items based on meaning rather than keywords.",
    "Agentic systems are architectures where multiple autonomous LLM agents collaborate, each with specialized roles, memory, and tools. They enable complex workflows—like planning, research, coding, or multi-step reasoning—by letting agents communicate, delegate tasks, and make decisions."
]

In [6]:
documents = [Document(page_content=text) for text in sample_texts]

In [7]:
#vector store creation
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(k=3)

## Agentic Function

In [8]:
def decide_retrieval(state:AgentState)->AgentState:
    """ 
    Decide if we need to retrieve documents based on the question
    """
    question = state["question"]

    #Simple heuristic: if the question contains certain keywords, retrieve
    retrieval_keywords = ["what", "how", "explain", "describe", "tell me"]
    needs_retrieval = any(keyword in question.lower() for keyword in retrieval_keywords)

    return {**state, "needs_retrieval": needs_retrieval} #Takes everything from the old state, copy it into a new dictionary, and update the needs_retrieval field with the new value.

In [11]:
def retrieve_documents(state: AgentState) -> AgentState:
    """ 
    Retrieve relevant documents based on the question
    """
    question = state["question"]
    documents = retriever.invoke(question)

    return {**state, "documents": documents}

In [12]:
def generate_answer(state: AgentState) -> AgentState:
    """ 
    Generate an answer using the retrieved documents or direct response
    """
    question = state["question"]
    documents = state.get("documents", [])

    if documents:
        #RAG approach: use documents as context
        context = "\n\n.join([doc.page_content for doc in documents])"
        prompt = f"""Based on the following context, answer the question
        Context: {context}
        Question: {question}
        Answer:
        """
    else:
        #Direct response without retrieval
        prompt = f"Answer the following question: {question}"

    response = llm.invoke(prompt)
    answer = response.content

    return {**state, "answer":answer}

## Conditional logic